# SEER Breast Cancer 2010–2017 — Model Preparation (Stages 4–6)

Implements `Documents/SEER_BC_Revision_Spec_v4.md` §5, Stages 4–6. It starts from the cohort produced by `seer_breast_2010-2017_pre-process_v4.ipynb`, which already covers Stages 1–2 and most of Stage 3: eligibility cascade, outcome `status`, sentinel recodes, stage/TNM harmonization, and the era-leak probe. Nothing from v4 is recomputed here.

**Steps**
1. **Stage 3 (remaining):** fix the 28 predictors, clinical grouping of rare levels, reference levels and numeric checks. Output: `cohort.parquet` and table S1.
2. **Stage 4 — Labeling:** a four-way category and label Y per horizon. Output: `labels.parquet`.
3. **Stage 5 — Split assignment:** 3 × 5-fold outer CV, plus an 80/20 inner split within each outer-train fold. Output: `folds.parquet`.
4. **Stage 6 — In-fold transformer:** imputation, `log1p`, scaling and one-hot encoding with a frozen column order. Outputs: `design_transformer.joblib` and `transformer_spec.json`.
5. **Year-proxy audit** on the final design matrix, run inside a fold.
6. **Manifest:** SHA-256 hashes of every artifact, so all downstream runs can verify they use the same files.

**Guiding rule (spec §5):** anything deterministic and label-free runs once here on the full cohort. Anything estimated from data (imputer, scaler, Ĝ, models) is fitted inside each fold.

In [ ]:
from pathlib import Path
import config
import hashlib
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

NOTEBOOK_DIR = config.WORK_DIR    # all outputs below this folder (config.py)
INPUT_DIR = NOTEBOOK_DIR / "outputs_model_ready"   # written by the v4 preprocessing notebook
OUTPUT_DIR = NOTEBOOK_DIR / "outputs_model_prep"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_REPEATS = 3                 # outer CV repeats (spec §5 Stage 5: decided now, persisted)
N_FOLDS = 5
INNER_VAL_FRAC = 0.20         # inner-val share of each outer-train fold
FOLLOWUP_BAND_MONTHS = 24     # follow-up band width for the split strata
FOLD_BALANCE_TOL_PP = 0.5     # max |status % in partition - cohort %|, percentage points
RARE_LEVEL_PCT = 0.5          # non-reference levels below this must be grouped (spec §5 3d)
YEAR_PROXY_AUC_MAX = 0.65     # year-proxy audit threshold (spec §5 3c: "well above ~0.6")

metadata_v4 = json.loads((INPUT_DIR / "metadata.json").read_text(encoding="utf-8"))
HORIZONS = metadata_v4["horizons_months"]
SEED = metadata_v4["random_state"]
print(f"Horizons: {HORIZONS} | seed: {SEED}")


def sha256(path: Path) -> str:
    """SHA-256 of a file; downstream runs compare it with prep_manifest.json."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

## Predictors and encoding design (Stage 3, remaining points)

v4 already recoded the sentinels (3a), harmonized stage/TNM (3c), and ran the era-leak probe. The Collaborative Stage site-specific factors (3b; SSF 1–7 and 15) are in the extract but are not used: they end with 2015 diagnoses, and receptor status comes from the ER, PR and Breast Subtype recodes, which are coded consistently across 2010–2017. This section finishes 3d–3f.

**28 prespecified predictors (6 numeric, 22 categorical).** This is v4's diagnosis-only set minus four exact or near duplicates, which would make the single design matrix singular for logistic regression and Fine–Gray:

| Dropped | Reason |
|---|---|
| `Primary Site` (code) | Same variable as `Primary Site - labeled` |
| `Derived HER2 Recode` | HER2+ = sum of the two HER2+ `Breast Subtype` levels (exact linear combination) |
| `Histology_broad_groupings`, `Histology_Group` | Coarser versions of `Histologic_Type`; after grouping, "Mixed ductal & lobular" is identical to code 8522 |

**Categorical design (3d).**
- Every raw level is mapped explicitly to a clinical level; an unmapped level stops the notebook.
- Rare levels (< 0.5% of the cohort) are merged with a clinically adjacent level, for example Grade IV → Grade III–IV, axillary tail → upper-outer quadrant, and T4a/T4c/T4 NOS together.
- The reference level is fixed a priori and placed first, so one-hot encoding drops it.
- Explicit "Unknown" levels and the rare level of a binary variable are kept even when below 0.5%.
- N0, N0(i−) and N0(mol−) are merged: all are pN0 in AJCC, and the N0/N0(i−) split is a reporting-practice difference that drifts with era.
- In `nodes_positive_desc`, "none examined" is merged with "unknown". The same patients are already flagged by `nodes_examined_desc` = "None examined" (SEER codes 00/98 always go together), so a separate level would be a duplicate column.

**Numerics (3a, 3e).**
- Range checks guard against any sentinel code surviving as a quantity (pitfall 11).
- `log1p` is applied to tumor size and the three node counts.
- Each numeric with missing values has a paired `_desc` category that already encodes *why* it is missing. The notebook asserts `NaN ⇔ non-count category`, so no extra missing-indicator columns are added; they would duplicate those dummies exactly. Any other numeric that has NaN gets an indicator automatically.

In [ ]:
NUMERIC_VARS = {                      # short name -> v4 column
    "age_years": "age_years",
    "income_ord": "income_ord",
    "tumor_size_mm": "tumor_size_over_time",
    "nodes_examined": "regional_nodes_examined_number",
    "nodes_positive": "regional_nodes_positive_number",
    "ln_axillary_positive": "ln_positive_axillary_number",
}
LOG1P_VARS = ["tumor_size_mm", "nodes_examined", "nodes_positive", "ln_axillary_positive"]
NUMERIC_RANGES = {                    # plausible ranges after v4's sentinel recoding
    "age_years": (20, 90),
    "income_ord": (0, 250_000),
    "tumor_size_mm": (1, 400),
    "nodes_examined": (0, 90),
    "nodes_positive": (0, 90),
    "ln_axillary_positive": (0, 90),
}

YES_NO = {"No": "No", "Yes": "Yes"}
POS_NEG = {"Positive": "Positive", "Negative": "Negative"}
T4_OTHER = "T4a, T4c or T4 NOS"
N1_OTHER = "N1b, N1c or N1 NOS"
N2_OTHER = "N2b or N2 NOS"
N3_OTHER = "N3b, N3c or N3 NOS"

# "map": raw value -> level (None = identity); "unknown_levels": exempt from the rare-level rule
CATEGORICAL_SPEC = {
    "race": {
        "source": "Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)",
        "reference": "Non-Hispanic White",
        "map": None,
    },
    "marital": {
        "source": "Marital status at diagnosis",
        "reference": "Married or domestic partner",
        "map": {
            "Married (including common law)": "Married or domestic partner",
            "Unmarried or Domestic Partner": "Married or domestic partner",   # 0.4%
            "Single (never married)": "Single",
            "Widowed": "Widowed",
            "Divorced": "Divorced",
            "Separated": "Separated",
        },
    },
    "primary_site": {
        "source": "Primary Site - labeled",
        "reference": "Upper-outer or axillary tail",
        "map": {
            "C50.0-Nipple": "Nipple or central",                                # 0.3%
            "C50.1-Central portion of breast": "Nipple or central",
            "C50.2-Upper-inner quadrant of breast": "Upper-inner",
            "C50.3-Lower-inner quadrant of breast": "Lower-inner",
            "C50.4-Upper-outer quadrant of breast": "Upper-outer or axillary tail",
            "C50.5-Lower-outer quadrant of breast": "Lower-outer",
            "C50.6-Axillary tail of breast": "Upper-outer or axillary tail",    # 0.45%
            "C50.8-Overlapping lesion of breast": "Overlapping",
            "C50.9-Breast, NOS": "Breast NOS",
        },
    },
    "laterality": {
        "source": "Laterality",
        "reference": "Left",
        "map": {"Left - origin of primary": "Left", "Right - origin of primary": "Right"},
    },
    "grade": {
        "source": "Grade Recode (thru 2017)",
        "reference": "Grade I",
        "map": {
            "Well differentiated; Grade I": "Grade I",
            "Moderately differentiated; Grade II": "Grade II",
            "Poorly differentiated; Grade III": "Grade III-IV",
            "Undifferentiated; anaplastic; Grade IV": "Grade III-IV",            # 0.06%
        },
    },
    "histology": {
        "source": "Histologic_Type",
        "reference": "Ductal NOS (8500)",
        "map": None,                                                            # built below from codes
    },
    "stage_group": {
        "source": "stage_group_harmonized",
        "reference": "IA",
        "map": {
            "IA": "IA", "IB": "IB", "IIA": "IIA", "IIB": "IIB",
            "IIIA": "IIIA", "III_NOS": "IIIA",                                  # 10 rows
            "IIIB": "IIIB", "IIIC": "IIIC", "IV": "IV",
        },
    },
    "t_stage": {
        "source": "t_stage_harmonized",
        "reference": "T1c or T1 NOS",
        "map": {
            "T0": "T0 or T1mic", "T1mic": "T0 or T1mic",
            "T1a": "T1a", "T1b": "T1b", "T1c": "T1c or T1 NOS", "T1_NOS": "T1c or T1 NOS",
            "T2": "T2", "T3": "T3",
            "T4a": T4_OTHER, "T4c": T4_OTHER, "T4_NOS": T4_OTHER,
            "T4b": "T4b", "T4d": "T4d",
        },
    },
    "n_stage": {
        "source": "n_stage_harmonized",
        "reference": "N0",
        "map": {
            "N0": "N0", "N0(i-)": "N0", "N0(mol-)": "N0",
            "N0(i+)": "N0(i+/mol+)", "N0(mol+)": "N0(i+/mol+)",
            "N1mi": "N1mi", "N1a": "N1a",
            "N1b": N1_OTHER, "N1c": N1_OTHER, "N1_NOS": N1_OTHER,
            "N2a": "N2a", "N2b": N2_OTHER, "N2_NOS": N2_OTHER,
            "N3a": "N3a", "N3b": N3_OTHER, "N3c": N3_OTHER, "N3_NOS": N3_OTHER,
        },
    },
    "m_stage": {
        "source": "m_stage_harmonized",
        "reference": "M0",
        "map": {"M0": "M0", "M0(i+)": "M0", "M1": "M1"},                        # cM0(i+) is M0
    },
    "subtype": {"source": "Breast Subtype (2010+)", "reference": "HR+/HER2-", "map": None},
    "er": {"source": "ER Status Recode Breast Cancer (1990+)", "reference": "Positive", "map": POS_NEG},
    "pr": {"source": "PR Status Recode Breast Cancer (1990+)", "reference": "Positive", "map": POS_NEG},
    "ln_axillary_desc": {
        "source": "ln_positive_axillary_desc",
        "reference": "All negative",
        "map": {
            "All axillary level I-II nodes examined negative": "All negative",
            "Exact number of positive axillary level I-II nodes": "Exact count",
            "90 or more positive axillary level I-II nodes": "Exact count",
            "Positive aspiration or needle core biopsy": "Positive, count unknown",
            "Positive nodes, number unspecified": "Positive, count unknown",
            "No axillary nodes examined": "Unknown or not applicable",
            "Unknown or not applicable": "Unknown or not applicable",
        },
        "unknown_levels": ["Unknown or not applicable"],
    },
    "nodes_examined_desc": {
        "source": "regional_nodes_examined_desc",
        "reference": "Exact count",
        "map": {
            "Exact number of nodes examined": "Exact count",
            "90 or more nodes were examined": "Exact count",
            "No nodes were examined": "None examined",
            "No regional nodes removed, aspiration performed": "Aspiration only",
            "Nodes surgically removed, number unknown": "Number unknown",
            "Removal documented as dissection, number unknown": "Number unknown",
            "Removal documented as sampling, number unknown": "Number unknown",
            "Unknown whether nodes were examined": "Number unknown",
        },
        "unknown_levels": ["Number unknown"],
    },
    "nodes_positive_desc": {
        "source": "regional_nodes_positive_desc",
        "reference": "All negative",
        "map": {
            "All nodes examined are negative": "All negative",
            "Exact number of nodes positive": "Exact count",
            "90 or more nodes are positive": "Exact count",
            # "None examined" is identical to nodes_examined_desc = "None examined" (codes 00/98),
            # so it is merged with Unknown here to avoid a duplicate column
            "No nodes were examined": "Not examined or unknown",
            "Unknown whether nodes are positive": "Not examined or unknown",
            "Positive aspiration of lymph node(s) was performed": "Positive, count unknown",
            "Positive nodes documented, number unspecified": "Positive, count unknown",
        },
        "unknown_levels": ["Not examined or unknown"],
    },
    "tumor_size_measured": {
        # Microscopic focus / no primary / diffuse (998) carry their meaning in t_stage (T1mic, T0, T4d)
        "source": "tumor_size_desc",
        "reference": "Measured",
        "map": {
            "Exact size in mm": "Measured",
            "Microscopic focus": "Not measured",
            "No evidence of primary tumor": "Not measured",
            "Site-specific code": "Not measured",
            "Unknown or size unreasonable": "Not measured",
        },
    },
    "mets_bone": {"source": "SEER Combined Mets at DX-bone (2010+)", "reference": "No", "map": YES_NO},
    "mets_brain": {"source": "SEER Combined Mets at DX-brain (2010+)", "reference": "No", "map": YES_NO},
    "mets_liver": {"source": "SEER Combined Mets at DX-liver (2010+)", "reference": "No", "map": YES_NO},
    "mets_lung": {"source": "SEER Combined Mets at DX-lung (2010+)", "reference": "No", "map": YES_NO},
    "rural_urban": {
        "source": "rural_urban_continuum",
        "reference": "Metro >=1M",
        "map": {
            "Counties in metropolitan areas ge 1 million pop": "Metro >=1M",
            "Counties in metropolitan areas of 250,000 to 1 million pop": "Metro 250k-1M",
            "Counties in metropolitan areas of lt 250 thousand pop": "Metro <250k",
            "Nonmetropolitan counties adjacent to a metropolitan area": "Nonmetro, adjacent",
            "Nonmetropolitan counties not adjacent to a metropolitan area": "Nonmetro, not adjacent",
            "Unknown/missing/no match (Alaska or Hawaii - Entire State)": "Unknown",
            "Unknown/missing/no match/Not 1990-2024": "Unknown",
            "Unknown": "Unknown",
        },
        "unknown_levels": ["Unknown"],
    },
}

# Histology: frequent codes (>= 0.5%) kept; rare codes grouped by v4's Histology_Group
HISTOLOGY_KEEP = {
    "8500": "Ductal NOS (8500)",
    "8520": "Lobular (8520, other lobular)",
    "8522": "Ductal and lobular (8522)",
    "8523": "Ductal mixed with other (8523)",
    "8480": "Mucinous (8480)",
    "8211": "Tubular (8211)",
}
HISTOLOGY_RARE_BY_GROUP = {
    "Invasive Ductal": "Other ductal",
    "Invasive Lobular": "Lobular (8520, other lobular)",   # other lobular codes total 0.34%
    "Special Type Invasive": "Other special type or unclassified",
    "Other / Unclassified": "Other special type or unclassified",
}

# Numeric -> (paired description variable, levels in which the numeric holds a true value)
NUMERIC_DESC_PAIRS = {
    "tumor_size_mm": ("tumor_size_measured", {"Measured"}),
    "nodes_examined": ("nodes_examined_desc", {"Exact count", "None examined"}),
    "nodes_positive": ("nodes_positive_desc", {"All negative", "Exact count"}),
    "ln_axillary_positive": ("ln_axillary_desc", {"All negative", "Exact count"}),
}

assert len(NUMERIC_VARS) + len(CATEGORICAL_SPEC) == 28, "predictor count changed"

## Load the v4 cohort and build `cohort.parquet`

In [ ]:
source_cols = (
    ["Patient ID", "Year of diagnosis", "time", "event_cr", "Vital status recode (study cutoff used)", "Histology_Group"]
    + list(NUMERIC_VARS.values())
    + [spec["source"] for spec in CATEGORICAL_SPEC.values()]
)
raw = pd.read_csv(
    INPUT_DIR / "model_df.csv", usecols=sorted(set(source_cols)), dtype=str, keep_default_na=False
)

# Input contract for the file handoff (v4 asserts the same on its side)
V = pd.to_numeric(raw["time"])
status = pd.to_numeric(raw["event_cr"]).astype(int)
assert raw["Patient ID"].is_unique, "Patient ID is not unique"
assert (V == V.round()).all() and (V > 0).all(), "V must be a positive integer (months)"
assert set(status.unique()) <= {0, 1, 2}
assert (status.eq(0) == raw["Vital status recode (study cutoff used)"].eq("Alive")).all(), "status 0 must equal vital status Alive"

cohort = pd.DataFrame({
    "patient_id": raw["Patient ID"],
    "V": V.astype("int16"),
    "status": status.astype("int8"),
    "year_dx": pd.to_numeric(raw["Year of diagnosis"]).astype("int16"),   # censoring model only, never in X
})

for name, col in NUMERIC_VARS.items():
    cohort[name] = pd.to_numeric(raw[col].replace("", np.nan))
    lo, hi = NUMERIC_RANGES[name]
    values = cohort[name].dropna()
    assert values.between(lo, hi).all(), f"{name}: values outside [{lo}, {hi}] -- a sentinel code may have survived"

code_to_group = raw.groupby("Histologic_Type")["Histology_Group"].agg(lambda s: s.unique())
assert code_to_group.map(len).eq(1).all(), "a histology code maps to several Histology_Group values"
CATEGORICAL_SPEC["histology"]["map"] = {
    code: HISTOLOGY_KEEP[code] if code in HISTOLOGY_KEEP else HISTOLOGY_RARE_BY_GROUP[groups[0]]
    for code, groups in code_to_group.items()
}

for name, spec in CATEGORICAL_SPEC.items():
    values = raw[spec["source"]]
    mapped = values if spec["map"] is None else values.map(spec["map"])
    unmapped = sorted(values[mapped.isna()].unique())
    assert not unmapped, f"{name}: unmapped raw levels {unmapped}"
    cohort[name] = mapped

for num, (desc, value_levels) in NUMERIC_DESC_PAIRS.items():
    assert (cohort[num].isna() == ~cohort[desc].isin(value_levels)).all(), (
        f"{num} is missing outside the non-count levels of {desc}"
    )
MISSING_INDICATOR_VARS = [
    name for name in NUMERIC_VARS if cohort[name].isna().any() and name not in NUMERIC_DESC_PAIRS
]
print(f"Numerics needing a missing indicator: {MISSING_INDICATOR_VARS or 'none'}")

X_COLS = list(NUMERIC_VARS) + list(CATEGORICAL_SPEC)
assert "year_dx" not in X_COLS
print(f"Cohort: {len(cohort):,} patients, {len(X_COLS)} predictors")

### Level frequencies, reference levels and table S1

Each variable's levels are ordered with the reference first, then by frequency. That order is frozen and used by the one-hot encoder. Non-reference levels must reach 0.5%, unless they are an explicit Unknown level or the rare level of a binary variable.

In [ ]:
CATEGORY_LEVELS = {}
s1_rows = []
for name, spec in CATEGORICAL_SPEC.items():
    counts = cohort[name].value_counts()
    ref = spec["reference"]
    assert ref in counts.index, f"{name}: reference level '{ref}' not present"
    levels = [ref] + [lvl for lvl in counts.index if lvl != ref]
    CATEGORY_LEVELS[name] = levels

    inverse = {}
    for raw_value, level in (spec["map"] or {lvl: lvl for lvl in levels}).items():
        inverse.setdefault(level, []).append(raw_value)
    for lvl in levels:
        pct = counts[lvl] / len(cohort) * 100
        exempt = len(levels) == 2 or lvl in spec.get("unknown_levels", [])
        assert lvl == ref or exempt or pct >= RARE_LEVEL_PCT, f"{name}: level '{lvl}' is rare ({pct:.2f}%)"
        s1_rows.append({
            "variable": name,
            "source_column": spec["source"],
            "level": lvl,
            "reference": lvl == ref,
            "n": int(counts[lvl]),
            "pct": round(pct, 2),
            "raw_values": "; ".join(sorted(inverse.get(lvl, [lvl]))),
        })

table_s1 = pd.DataFrame(s1_rows)
table_s1.to_csv(OUTPUT_DIR / "table_S1_harmonization.csv", index=False)
display(table_s1.drop(columns="raw_values"))

cohort = cohort[["patient_id"] + X_COLS + ["V", "status", "year_dx"]]
cohort.to_parquet(OUTPUT_DIR / "cohort.parquet", index=False)
print(f"Saved cohort.parquet: {cohort.shape}")

## Stage 4 — Labeling

For each τ in {36, 60, 120}, every patient falls in exactly one category (spec §5 Stage 4):

| Category | Condition | Y |
|---|---|---|
| `bc_death` | status = 1 and V ≤ τ | 1 |
| `competing_death` | status = 2 and V ≤ τ | 0 |
| `known_alive` | V ≥ τ and no death by τ | 0 |
| `indeterminate` | status = 0 and V < τ | −1 |

- **Boundary:** a patient alive with `V == τ` was observed through τ, so they are `known_alive`.
- **How each cell uses the labels:**
  - Zero: indeterminate patients get Y = 0 and weight 1.
  - Discard: indeterminate patients are dropped.
  - IPCW: indeterminate patients get weight 0.
- Competing deaths are Y = 0 in every cell.
- No weights are computed here; they need Ĝ and are computed in-fold.

**Checks:**
- Categories are exhaustive and mutually exclusive.
- Counts reconcile with table T1 from v4.
- Labels are coherent across horizons: a death stays a death at later horizons, an indeterminate patient stays indeterminate, and known-alive at a later τ implies known-alive earlier.

In [ ]:
LABEL_CATEGORIES = ["bc_death", "competing_death", "known_alive", "indeterminate"]
Y_BY_CATEGORY = {"bc_death": 1, "competing_death": 0, "known_alive": 0, "indeterminate": -1}


def label_categories(V, status, tau):
    """Four-way horizon category (spec §5 Stage 4)."""
    bc = (status == 1) & (V <= tau)
    comp = (status == 2) & (V <= tau)
    alive = (V >= tau) & ~bc & ~comp
    indet = (status == 0) & (V < tau)
    n_true = bc.astype(int) + comp + alive + indet
    assert (n_true == 1).all(), f"tau={tau}: categories are not exhaustive and mutually exclusive"
    return np.select([bc, comp, alive, indet], LABEL_CATEGORIES, default="")


V_arr = cohort["V"].to_numpy()
status_arr = cohort["status"].to_numpy()
labels = pd.DataFrame({"patient_id": cohort["patient_id"]})
for tau in HORIZONS:
    category = label_categories(V_arr, status_arr, tau)
    labels[f"category_{tau}"] = pd.Categorical(category, categories=LABEL_CATEGORIES)
    labels[f"y_{tau}"] = pd.Series(category).map(Y_BY_CATEGORY).astype("int8").to_numpy()

counts = pd.DataFrame(
    {tau: labels[f"category_{tau}"].value_counts().reindex(LABEL_CATEGORIES) for tau in HORIZONS}
).T.rename_axis("horizon_months")
counts["n_determinable"] = counts[["bc_death", "competing_death", "known_alive"]].sum(axis=1)
counts["competing_deaths_recovered"] = counts["competing_death"]   # spec §3.1 required reporting
display(counts)

# Reconcile with table T1 from the v4 notebook
t1 = pd.read_csv(INPUT_DIR / "table_T1_by_horizon.csv", index_col="horizon_months")
for tau in HORIZONS:
    assert counts.loc[tau, "bc_death"] == t1.loc[tau, "bc_deaths"]
    assert counts.loc[tau, "competing_death"] == t1.loc[tau, "other_cause_deaths"]
    assert counts.loc[tau, "indeterminate"] == t1.loc[tau, "censored_before_tau"]
    assert counts.loc[tau, "n_determinable"] == t1.loc[tau, "n_determinable"]
print("Label counts reconcile with table T1.")

# Coherence across horizons (tau_a < tau_b)
for tau_a, tau_b in zip(HORIZONS[:-1], HORIZONS[1:]):
    cat_a, cat_b = labels[f"category_{tau_a}"], labels[f"category_{tau_b}"]
    for cat in ["bc_death", "competing_death", "indeterminate"]:
        assert (cat_b[cat_a == cat] == cat).all(), f"{cat} at {tau_a} is not {cat} at {tau_b}"
    assert (cat_a[cat_b == "known_alive"] == "known_alive").all(), f"known_alive at {tau_b} but not at {tau_a}"
print("Labels are coherent across horizons.")

labels.to_parquet(OUTPUT_DIR / "labels.parquet", index=False)
print(f"Saved labels.parquet: {labels.shape}")

## Stage 5 — Split assignment

- **Strata:** `status` × follow-up band (V in 24-month bands). One assignment keeps the event and censoring structure balanced for all three horizons.
- **Outer CV:** 3 repeats × 5 folds, stratified. The same folds are used for every cell, model and horizon, so comparisons are paired (needed for Nadeau–Bengio and paired DeLong).
- **Inner split:** each outer-train fold is split 80/20 into inner-train and inner-val, with the same stratification.

| `inner_role` | Used for |
|---|---|
| `inner_train` + `inner_val` (= outer-train) | Fitting Ĝ, imputer, scaler |
| `inner_train` | Model fitting; SMOTE applied here only |
| `inner_val` | Hyperparameters and γ\* selection (no refit afterwards) |
| `test` | Final evaluation only |

`folds.parquet` has one row per patient × repeat × outer fold, with columns `patient_id, repeat, outer_fold, inner_role`. Here `outer_fold` is the index of the fold held out as test. Every downstream script reads this file and checks its hash; nothing re-splits.

In [ ]:
followup_band = cohort["V"] // FOLLOWUP_BAND_MONTHS
strata = cohort["status"].astype(str) + "|" + followup_band.astype(str)
stratum_sizes = strata.value_counts()
print(f"{len(stratum_sizes)} strata; smallest: {stratum_sizes.idxmin()} with {stratum_sizes.min():,} patients")
assert stratum_sizes.min() >= 2 * N_FOLDS, "a stratum is too small for the outer and inner splits"

patient_ids = cohort["patient_id"].to_numpy()
strata_arr = strata.to_numpy()
parts = []
for r in range(N_REPEATS):
    outer = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + r)
    for k, (train_idx, test_idx) in enumerate(outer.split(patient_ids, strata_arr)):
        inner_train_idx, inner_val_idx = train_test_split(
            train_idx,
            test_size=INNER_VAL_FRAC,
            stratify=strata_arr[train_idx],
            random_state=SEED + 1000 * (r + 1) + k,
        )
        role = np.empty(len(patient_ids), dtype=object)
        role[test_idx] = "test"
        role[inner_train_idx] = "inner_train"
        role[inner_val_idx] = "inner_val"
        parts.append(pd.DataFrame({"patient_id": patient_ids, "repeat": r, "outer_fold": k, "inner_role": role}))

folds = pd.concat(parts, ignore_index=True)
folds["repeat"] = folds["repeat"].astype("int8")
folds["outer_fold"] = folds["outer_fold"].astype("int8")
folds["inner_role"] = pd.Categorical(folds["inner_role"], categories=["inner_train", "inner_val", "test"])

# Every patient has a role in every split, and is tested exactly once per repeat
assert folds["inner_role"].notna().all()
test_counts = folds[folds["inner_role"] == "test"].groupby(["repeat", "patient_id"], observed=True).size()
assert len(test_counts) == len(cohort) * N_REPEATS and (test_counts == 1).all()

# Fold balance: status distribution per partition within tolerance of the cohort
status_by_patient = cohort.set_index("patient_id")["status"]
cohort_pct = status_by_patient.value_counts(normalize=True).sort_index() * 100
balance = (
    folds.assign(status=folds["patient_id"].map(status_by_patient))
    .groupby(["repeat", "outer_fold", "inner_role"], observed=True)["status"]
    .value_counts(normalize=True)
    .mul(100)
    .unstack("status")
)
max_dev = (balance - cohort_pct).abs().max().max()
print(f"Max deviation of status % from the cohort: {max_dev:.3f} pp (tolerance {FOLD_BALANCE_TOL_PP} pp)")
assert max_dev <= FOLD_BALANCE_TOL_PP

display(
    folds.groupby(["repeat", "inner_role"], observed=True).size().unstack("inner_role").div(N_FOLDS).round(0)
    .rename(columns=lambda c: f"mean n per fold: {c}")
)

folds.to_parquet(OUTPUT_DIR / "folds.parquet", index=False)
print(f"Saved folds.parquet: {folds.shape}")

## Stage 6 — In-fold transformer

`build_transformer()` returns an **unfitted** transformer. In every fold it is fitted on outer-train (`inner_role != "test"`) and then applied unchanged to inner-train, inner-val and test:

- **Numerics:** `log1p` (tumor size, node counts) → median imputation → standard scaling. Imputation comes before SMOTE, which cannot handle NaN.
- **Missing indicators:** only for numerics whose missingness is not already coded by a `_desc` variable (none in this cohort).
- **Categoricals:** one-hot with the fixed level lists above; the reference level (first) is dropped. Nothing is learned from the data, so the columns are identical in every fold.

**Checks:**
- The output contains no NaN or infinite values.
- Fits on two different folds give the same column names in the same order.
- `year_dx` is absent from the features.
- The design matrix plus intercept has full column rank, so there are no exact duplicates.

In [ ]:
def build_transformer():
    """Unfitted design-matrix transformer (spec §5 Stage 6). Fit on outer-train only."""
    log_cols = [c for c in NUMERIC_VARS if c in LOG1P_VARS]
    lin_cols = [c for c in NUMERIC_VARS if c not in LOG1P_VARS]
    cat_cols = list(CATEGORICAL_SPEC)
    steps = [
        ("num_log", Pipeline([
            ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), log_cols),
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), lin_cols),
        ("cat", OneHotEncoder(
            categories=[CATEGORY_LEVELS[c] for c in cat_cols],
            drop="first",
            handle_unknown="error",
            sparse_output=False,
        ), cat_cols),
    ]
    if MISSING_INDICATOR_VARS:
        steps.append(("missing", MissingIndicator(features="all"), MISSING_INDICATOR_VARS))
    return ColumnTransformer(steps, remainder="drop", verbose_feature_names_out=True)


def partition(r, k):
    """inner_role of every cohort row for repeat r, outer fold k."""
    roles = folds.loc[(folds["repeat"] == r) & (folds["outer_fold"] == k)].set_index("patient_id")["inner_role"]
    return cohort["patient_id"].map(roles)


role = partition(0, 0)
outer_train = cohort[role.ne("test")]
test = cohort[role.eq("test")]

transformer = build_transformer().fit(outer_train[X_COLS])
X_train = transformer.transform(outer_train[X_COLS])
X_test = transformer.transform(test[X_COLS])
FEATURE_NAMES = list(transformer.get_feature_names_out())

assert np.isfinite(X_train).all() and np.isfinite(X_test).all(), "NaN/inf in the design matrix"
assert not any("year_dx" in f for f in FEATURE_NAMES), "year of diagnosis reached the design matrix"

other_fit = clone(build_transformer()).fit(cohort.loc[partition(N_REPEATS - 1, N_FOLDS - 1).ne("test"), X_COLS])
assert list(other_fit.get_feature_names_out()) == FEATURE_NAMES, "column order differs between folds"

rank = np.linalg.matrix_rank(np.column_stack([np.ones(len(X_train)), X_train]))
assert rank == X_train.shape[1] + 1, f"design matrix is rank-deficient ({rank} < {X_train.shape[1] + 1})"

print(f"Design matrix: {X_train.shape[1]} columns from {len(X_COLS)} predictors; full column rank.")
print(f"Outer-train {X_train.shape[0]:,} rows | test {X_test.shape[0]:,} rows")

## Year-proxy audit on the final design matrix (spec §5 3c, table S1)

v4's probe used an ad-hoc encoding. This one uses the actual design matrix and runs inside a fold: logistic regression predicts diagnosis period (2016–17 vs 2010–15), fitted on outer-train and scored on the test fold of repeat 0, fold 0. An AUC above `YEAR_PROXY_AUC_MAX` stops the notebook. The largest coefficients show which columns carry any remaining era signal.

In [ ]:
era_train = outer_train["year_dx"].ge(2016).astype(int)
era_test = test["year_dx"].ge(2016).astype(int)

probe = LogisticRegression(max_iter=5000)
probe.fit(X_train, era_train)
year_proxy_auc = roc_auc_score(era_test, probe.predict_proba(X_test)[:, 1])
print(f"Year-proxy AUC (test fold): {year_proxy_auc:.3f}  (threshold {YEAR_PROXY_AUC_MAX})")

top_coefs = (
    pd.Series(probe.coef_[0], index=FEATURE_NAMES)
    .sort_values(key=np.abs, ascending=False)
    .head(12)
    .round(3)
)
display(top_coefs.to_frame("coefficient"))
assert year_proxy_auc < YEAR_PROXY_AUC_MAX, "year-proxy AUC above threshold: locate and harmonize the proxy"

## Save the transformer, its spec and the manifest

In [ ]:
joblib.dump(build_transformer(), OUTPUT_DIR / "design_transformer.joblib")

transformer_spec = {
    "predictors": {"numeric": list(NUMERIC_VARS), "categorical": list(CATEGORICAL_SPEC)},
    "n_predictors": len(X_COLS),
    "source_columns": {**NUMERIC_VARS, **{k: v["source"] for k, v in CATEGORICAL_SPEC.items()}},
    "excluded_from_X": ["patient_id", "V", "status", "year_dx"],
    "dropped_duplicates": {
        "Primary Site": "code duplicate of Primary Site - labeled",
        "Derived HER2 Recode (2010+)": "exact linear combination of Breast Subtype levels",
        "Histology_broad_groupings": "coarser version of Histologic_Type",
        "Histology_Group": "coarser version of Histologic_Type (used only to group rare codes)",
    },
    "sentinel_recodes": (
        "Done in v4: node codes 95-99 and tumor size 000/990/998/999 -> NaN in the numeric column, "
        "meaning kept in the paired _desc category."
    ),
    "numeric_ranges_checked": NUMERIC_RANGES,
    "log1p": LOG1P_VARS,
    "imputation": "median, fitted on outer-train",
    "missing_indicators": MISSING_INDICATOR_VARS,
    "numeric_desc_pairs": {k: [v[0], sorted(v[1])] for k, v in NUMERIC_DESC_PAIRS.items()},
    "scaling": "StandardScaler on numerics, fitted on outer-train",
    "one_hot": "fixed level lists, reference (first level) dropped",
    "reference_levels": {k: v["reference"] for k, v in CATEGORICAL_SPEC.items()},
    "category_levels": CATEGORY_LEVELS,
    "grouping_maps": {k: v["map"] for k, v in CATEGORICAL_SPEC.items() if v["map"] is not None},
    "rare_level_pct": RARE_LEVEL_PCT,
    "feature_names": FEATURE_NAMES,
    "year_proxy_auc": round(float(year_proxy_auc), 4),
    "year_proxy_auc_max": YEAR_PROXY_AUC_MAX,
}
(OUTPUT_DIR / "transformer_spec.json").write_text(json.dumps(transformer_spec, indent=2), encoding="utf-8")

artifacts = ["cohort.parquet", "labels.parquet", "folds.parquet", "design_transformer.joblib",
             "transformer_spec.json", "table_S1_harmonization.csv"]
manifest = {
    "source_notebook": "02_model_prep.ipynb",
    "input": {"model_df.csv": sha256(INPUT_DIR / "model_df.csv")},
    "sha256": {name: sha256(OUTPUT_DIR / name) for name in artifacts},
    "n_patients": int(len(cohort)),
    "horizons_months": HORIZONS,
    "seed": SEED,
    "cv": {
        "n_repeats": N_REPEATS,
        "n_folds": N_FOLDS,
        "inner_val_frac": INNER_VAL_FRAC,
        "strata": f"status x floor(V / {FOLLOWUP_BAND_MONTHS})",
        "outer_train": "inner_role != 'test'",
    },
    "labels": {"y": Y_BY_CATEGORY, "competing_deaths": "Y = 0 in all cells"},
    "censoring_model": {
        "event": "status == 0 (loss to follow-up / administrative end); both death types censored",
        "covariates": "design matrix X (78 columns from the 28 predictors; encoder fitted on outer-train); year_dx not used (positivity at 120 months)",
        "weight_convention": "left limit G(t-): evaluate at t - 0.5 for integer months; deaths before censorings at ties",
        "fit_on": "outer-train only",
    },
}
(OUTPUT_DIR / "prep_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

for name, digest in manifest["sha256"].items():
    print(f"{name:<28} {digest[:16]}...")
print(f"\nOutputs saved to: {OUTPUT_DIR}")

## Using these outputs in the modeling notebook

```python
manifest = json.loads((PREP_DIR / "prep_manifest.json").read_text())
for name, digest in manifest["sha256"].items():          # same files in every run
    assert sha256(PREP_DIR / name) == digest, f"{name} changed"

cohort = pd.read_parquet(PREP_DIR / "cohort.parquet")
labels = pd.read_parquet(PREP_DIR / "labels.parquet")
folds = pd.read_parquet(PREP_DIR / "folds.parquet")
template = joblib.load(PREP_DIR / "design_transformer.joblib")

for (r, k), split in folds.groupby(["repeat", "outer_fold"]):
    role = cohort["patient_id"].map(split.set_index("patient_id")["inner_role"])
    transformer = clone(template).fit(cohort.loc[role != "test", X_COLS])   # outer-train only
    # fit G on outer-train (year_dx, age_years) -> weights -> cell rule -> model on inner_train ...
```

**Open points for the modeling stage:**
- **SMOTE on one-hot columns** creates fractional dummy values. Use SMOTENC on the un-encoded categoricals, or state how dummies are handled.
- **Near-duplicates kept:** `stage_group = IV` and `m_stage = M1` agree for all but 21 patients. Use penalized logistic regression; watch Fine–Gray standard errors.